# Research 43 — Fold-safe one-hot LightGBM with VIP × region

**Hypothesis:** one-hot categorical splits may complement native LightGBM categorical handling. Use the balanced-weight VIP × region feature recipe and seed-42 three-fold OOF protocol. Encode categories using each training fold only; reindex validation dummies to training columns. This is a single encoding hypothesis, not a search. Test is not loaded.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def make_features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (num, den) in ratios.items():
        x[name] = x[num] / (x[den].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
    region = x["region"].fillna("__MISSING__").astype(str)
    x["vip_region"] = vip + "__" + region
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = make_features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
p_candidate = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    x_fit = X.iloc[fit_idx].copy()
    x_valid = X.iloc[valid_idx].copy()
    x_fit = pd.get_dummies(x_fit, columns=cat_cols, dummy_na=True, dtype=np.uint8)
    x_valid = pd.get_dummies(x_valid, columns=cat_cols, dummy_na=True, dtype=np.uint8)
    x_valid = x_valid.reindex(columns=x_fit.columns, fill_value=0)
    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx])
    p_candidate[valid_idx] = model.predict_proba(x_valid)
    score = f1_score(y[valid_idx], p_candidate[valid_idx].argmax(1), average="macro")
    print(f"Fold {fold}: macro F1={score:.6f}; one-hot features={x_fit.shape[1]}", flush=True)
pred = p_candidate.argmax(axis=1)
display(
    pd.DataFrame(
        [
            {
                "candidate": "Fold-safe one-hot LightGBM VIP × region",
                "macro_f1": f1_score(y, pred, average="macro"),
                "weighted_f1": f1_score(y, pred, average="weighted"),
            }
        ]
    ).round(6)
)
pd.DataFrame(p_candidate, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-lightgbm-onehot-vip-region-oof.csv", index=False
)
print(f"Elapsed seconds={perf_counter() - started:.1f}. Train OOF only; Test not loaded.")

Fold 1: macro F1=0.742035; one-hot features=127


Fold 2: macro F1=0.744263; one-hot features=127


Fold 3: macro F1=0.746591; one-hot features=127


,candidate,macro_f1,weighted_f1
0,Fold-safe one-hot LightGBM VIP × region,0.744308,0.756069


Elapsed seconds=103.6. Train OOF only; Test not loaded.
